# Genotype–Phenotype Heterogeneity and Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant schema URL:**  
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic information
print(metadata.name)
print(metadata.description)
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**The Croissant schema structurally defines record sets and fields. We will use their `@id` for subsequent referencing.**

In [ ]:
record_sets = dataset.record_sets()
print("Available record sets (@id and name):")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '[no name]')}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    @id: {f['@id']} | name: {f.get('name', '[no name]')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set `@id`s identified above, and reference columns/fields by `@id`. For demo purposes, we will extract all record sets present.

In [ ]:
dataframes = {}

# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Extracting the following record sets:", record_set_ids)

for record_set_id in record_set_ids:
    # Load records for each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

### Example: Filtering, Normalization, and Grouping
We will demonstrate using the first available record set and its numeric field(s). All references are made using `@id`.

In [ ]:
# Choose a record set and fields for analysis
if record_set_ids:
    record_set_id = record_set_ids[0]  # Pick the first record set
    df = dataframes[record_set_id]
    
    # Find numeric fields for demonstration
    rs_metadata = next((r for r in record_sets if r['@id'] == record_set_id), None)
    numeric_field_id = None
    group_field_id = None
    if rs_metadata:
        for field in rs_metadata.get('fields', []):
            field_type = field.get('dataType', '').lower()
            if ('float' in field_type or 'integer' in field_type or 'number' in field_type) and field['@id'] in df.columns:
                numeric_field_id = field['@id']
            if ('text' in field_type or 'string' in field_type or 'boolean' in field_type) and field['@id'] in df.columns:
                group_field_id = field['@id']
            if numeric_field_id and group_field_id:
                break

    # Fallback demo: use columns if not found
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not group_field_id:
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]):
                group_field_id = col
                break

    print(f"Using numeric field @id: {numeric_field_id}")
    print(f"Using group field @id: {group_field_id}")

    # Filter: select rows where numeric field value > threshold
    threshold = 10
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group field (if available)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we create a simple histogram for a numeric field, as well as a scatter plot if enough numeric columns exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If numeric field chosen above and dataframe exists...
if record_set_ids and numeric_field_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_field_id in df.columns:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field_id], bins=10, kde=True)
        plt.title(f"Distribution of {numeric_field_id} in record set {record_set_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Scatter plot between first two numeric columns
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.title(f"Scatter Plot: {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 dataset using `mlcroissant` from its Croissant schema URL.
- All operations referenced record set, field, and column entities by their `@id`, ensuring consistent entity handling.
- The data overview revealed several record sets with rich clinical, imaging, and genetic features from three female patients.
- Data extraction and EDA demonstrated filtering, normalization, and groupings using column `@id`s.
- Visualizations provided a quick view of feature distributions and relationships, useful for further clinical or genomic correlation analysis.

As the dataset is small (N=3) and highly structured, it is best suited for demonstration, replication, genotypic/phenotypic correlation, and illustrative medical research rather than AI training. Always reference entities using their `@id` for robust and reproducible data science workflows.